<a href="https://colab.research.google.com/github/partha-pkp/data-mining-drug-discovery/blob/main/avanie/Logistic_Screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install liac-arff

import pandas as pd
import numpy as np
from scipy.io import arff

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

  Preparing metadata (setup.py) ... done


In [ ]:
from google.colab import files

uploaded = files.upload()
ARFF_FILE = list(uploaded.keys())[0]

data, meta = arff.loadarff(ARFF_FILE)
df = pd.DataFrame(data)

TARGET_COL = "senolytic"
df[TARGET_COL] = df[TARGET_COL].str.decode("utf-8").astype(int)

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

print("Loaded labeled data:", df.shape)
print("Positive count:", int(y.sum()))
print("Negative count:", int((y == 0).sum()))

Saving input_data_no_smiles.arff to input_data_no_smiles.arff
Loaded labeled data: (4866, 201)
Positive count: 103
Negative count: 4763


In [ ]:
final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000
    ))
])

final_model.fit(X, y)

print("Final model fit complete.")

Final model fit complete.


In [ ]:
uploaded = files.upload()
screen_file = list(uploaded.keys())[0]

screen_df = pd.read_csv(screen_file)
print(screen_df.shape)
print(screen_df.columns.tolist())
screen_df.head()

Saving list_of_compounds_for_computational_screening (1).csv to list_of_compounds_for_computational_screening (1).csv
(4340, 203)
['Name', 'Library', 'SMILES', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'ExactMolWt', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'FractionCSP3', 'HallKierAlpha', 'HeavyAtomCount', 'HeavyAtomMolWt', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'MaxAbsEStateIndex', 'MaxAbsPartialCharge', 'MaxEStateIndex', 'MaxPartialCharge', 'MinAbsEStateIndex', 'MinAbsPartialCharge', 'MinEStateIndex', 'MinPartialCharge', 'MolLogP', 'MolMR', 'MolWt', 'NHOHCount', 'NOCount', 'NumAliphaticCarbocycles', 'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumA

,Name,Library,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed
0,(-)-Arctigenin,Selleck,COC1=C(O)C=CC(=C1)CC2C(COC2=O)CC3=CC=C(OC)C(=C...,1.763934,816.314286,19.388541,15.728578,15.728578,13.049575,8.787575,...,0,0,0,0,0,0,0,0,0,0.753438
1,(-)-Blebbistatin,Targetmol 3338,Cc1ccc2N=C3N(CC[C@@]3(O)C(=O)c2c1)c1ccccc1,1.880486,797.989385,15.319626,12.282905,12.282905,10.593172,7.390657,...,0,0,0,0,0,0,0,0,0,0.879020
2,(-)-Borneol,Selleck,CC1(C)C2CCC1(C)C(O)C2,2.396255,185.311799,8.276021,7.723234,7.723234,4.982999,4.663847,...,0,0,0,0,0,0,0,0,0,0.566800
3,(-)-Cotinine,Targetmol 3338,CN1[C@@H](CCC1=O)c1cccnc1,2.236360,309.960601,9.259149,7.603640,7.603640,6.287694,4.444035,...,0,0,0,0,0,0,0,0,0,0.647201
4,(-)-Epicatechin gallate,Selleck,OC1=CC2=C(CC(OC(=O)C3=CC(=C(O)C(=C3)O)O)C(O2)C...,1.721352,1196.568791,23.153972,16.258499,16.258499,15.133938,9.330270,...,0,0,0,0,0,0,0,0,0,0.235487


In [ ]:
possible_id_cols = ["SMILES", "compound_name", "Compound", "Name"]
id_cols = [c for c in possible_id_cols if c in screen_df.columns]

X_screen = screen_df.drop(columns=id_cols, errors="ignore").copy()

# force same feature order as training data
X_screen = X_screen[X.columns]

print("Screening features shape:", X_screen.shape)

Screening features shape: (4340, 200)


In [ ]:
threshold = 0.156214

screen_probs = final_model.predict_proba(X_screen)[:, 1]
screen_pred = (screen_probs >= threshold).astype(int)

results_df = screen_df.copy()
results_df["senolytic_probability"] = screen_probs
results_df["predicted_senolytic"] = screen_pred

results_df.head()

,Name,Library,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,...,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed,senolytic_probability,predicted_senolytic
0,(-)-Arctigenin,Selleck,COC1=C(O)C=CC(=C1)CC2C(COC2=O)CC3=CC=C(OC)C(=C...,1.763934,816.314286,19.388541,15.728578,15.728578,13.049575,8.787575,...,0,0,0,0,0,0,0,0.753438,1.0,1
1,(-)-Blebbistatin,Targetmol 3338,Cc1ccc2N=C3N(CC[C@@]3(O)C(=O)c2c1)c1ccccc1,1.880486,797.989385,15.319626,12.282905,12.282905,10.593172,7.390657,...,0,0,0,0,0,0,0,0.879020,1.0,1
2,(-)-Borneol,Selleck,CC1(C)C2CCC1(C)C(O)C2,2.396255,185.311799,8.276021,7.723234,7.723234,4.982999,4.663847,...,0,0,0,0,0,0,0,0.566800,1.0,1
3,(-)-Cotinine,Targetmol 3338,CN1[C@@H](CCC1=O)c1cccnc1,2.236360,309.960601,9.259149,7.603640,7.603640,6.287694,4.444035,...,0,0,0,0,0,0,0,0.647201,1.0,1
4,(-)-Epicatechin gallate,Selleck,OC1=CC2=C(CC(OC(=O)C3=CC(=C(O)C(=C3)O)O)C(O2)C...,1.721352,1196.568791,23.153972,16.258499,16.258499,15.133938,9.330270,...,0,0,0,0,0,0,0,0.235487,1.0,1


In [ ]:
# Count predictions
counts = results_df["predicted_senolytic"].value_counts()

num_positive = int(counts.get(1, 0))
num_negative = int(counts.get(0, 0))
total = len(results_df)

print(f"Total compounds: {total}")
print(f"Predicted senolytic (1): {num_positive}")
print(f"Predicted NOT senolytic (0): {num_negative}")

Total compounds: 4340
Predicted senolytic (1): 4279
Predicted NOT senolytic (0): 61


In [ ]:
results_df.to_csv("computational_screening_results.csv", index=False)
print("Saved computational_screening_results.csv")

Saved computational_screening_results.csv


In [ ]:
#approach 1
uploaded = files.upload()
paper21_file = list(uploaded.keys())[0]

paper21_df = pd.read_csv(paper21_file)
paper21_df.head()

Saving ResearchPaperPreds.csv to ResearchPaperPreds.csv


,Name,Library,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed
0,BMS 599626 2HCl (873837-23-1(HCl)),Targetmol 3338,Cl.Cl.Cc1c(NC(=O)OC[C@@H]2COCCN2)cn2ncnc(Nc3cc...,0.000002,1654.714519,26.769009,22.041661,23.674654,19.030010,12.582129,...,0,0,0,0,0,0,0,0,0,0.245145
1,BMS986142,Targetmol 3338,Cc1c(cccc1-n1c(=O)n(C)c2c(F)cccc2c1=O)-c1c(F)c...,1.647114,2081.914545,30.247830,23.639787,23.639787,19.806457,13.695529,...,0,0,0,0,0,0,0,0,0,0.295475
2,Ellagic acid,Targetmol 3338,Oc1c(O)c2c3c(c1)c(=O)oc1c3c(cc(O)c1O)c(=O)o2,2.554067,1101.712243,15.756630,10.576548,10.576548,10.396755,6.023873,...,0,0,0,0,0,0,0,0,0,0.216285
3,Everolimus,Targetmol 3338,CO[C@@H]1C[C@H](C[C@@H](C)[C@@H]2CC(=O)[C@H](C...,1.696795,1809.064720,50.293589,42.433193,42.433193,32.287588,25.006427,...,0,0,0,0,0,0,0,0,0,0.133924
4,gamma-Mangostin,Targetmol 3338,CC(=CCC1=C(C=C2C(=C1O)C(=O)C3=C(C(=C(C=C3O2)O)...,2.431298,1236.413795,21.455301,16.828966,16.828966,13.611969,9.262454,...,0,0,0,0,0,0,0,0,0,0.286723


In [ ]:
matched21 = results_df.merge(paper21_df, on="SMILES", how="inner")

print("Number of paper compounds found in screening set:", len(matched21))
print("Number predicted senolytic:", int(matched21["predicted_senolytic"].sum()))

matched21[["SMILES", "senolytic_probability", "predicted_senolytic"]]

Number of paper compounds found in screening set: 21
Number predicted senolytic: 21


,SMILES,senolytic_probability,predicted_senolytic
0,Cl.Cl.Cc1c(NC(=O)OC[C@@H]2COCCN2)cn2ncnc(Nc3cc...,1.0,1
1,Cc1c(cccc1-n1c(=O)n(C)c2c(F)cccc2c1=O)-c1c(F)c...,1.0,1
2,Oc1c(O)c2c3c(c1)c(=O)oc1c3c(cc(O)c1O)c(=O)o2,1.0,1
3,CO[C@@H]1C[C@H](C[C@@H](C)[C@@H]2CC(=O)[C@H](C...,1.0,1
4,CC(=CCC1=C(C=C2C(=C1O)C(=O)C3=C(C(=C(C=C3O2)O)...,1.0,1
5,COC1=CC(=CC=C1O)/C=C/C(=O)OC2CCC34CC35CCC6(C)C...,1.0,1
6,COc1cc(O)c2c(c1)oc(cc2=O)-c1ccc(OC)c(c1)-c1c(O...,1.0,1
7,CC(C)c1c(O)c(O)c(C=O)c2c(O)c(c(C)cc12)-c1c(C)c...,1.0,1
8,C1=CC(=CC=C1C2=C(C(=O)C3=C(O2)C(=C(C=C3O)O)O)O)O,1.0,1
9,OC1=CC(=C(C=C1)C2=C(O)C(=O)C3=C(O2)C=C(O)C=C3O)O,1.0,1


In [ ]:
#approach 2

In [ ]:
targets = {"ginkgetin", "periplocin", "oleandrin"}

name_col = None
for c in ["compound_name", "Compound", "Name"]:
    if c in results_df.columns:
        name_col = c
        break

if name_col is not None:
    three_hits = results_df[
        results_df[name_col].astype(str).str.strip().str.lower().isin(targets)
    ]
    display(three_hits[[name_col, "senolytic_probability", "predicted_senolytic"]])
else:
    print("No compound-name column found. Use exact SMILES instead.")

,Name,senolytic_probability,predicted_senolytic
1800,Ginkgetin,1.0,1
2911,Oleandrin,1.0,1
3078,Periplocin,1.0,1
